# LawBot tool-calling agent demo

Demonstrates case-law search, local glossary lookup, illustrative date arithmetic, RAG evidence retrieval, direct-answer routing, and per-call traces. Set `INDIANKANOON_TOKEN` for live Indian Kanoon search. Without a token, case search is explicitly labeled as a local cached-summary fallback.

Educational use only; not legal advice. Date calculations are illustrative and do not determine legal limitation periods.


In [ ]:
import os, sys, json
from pathlib import Path
roots = ([Path(os.environ["LAWBOT_PROJECT_DIR"]).expanduser()] if os.environ.get("LAWBOT_PROJECT_DIR") else [])
roots += [Path.cwd(), Path.cwd().parent, Path("/content/lawbot"), Path("/kaggle/working/lawbot")]
PROJECT_DIR = next((p.resolve() for p in roots if (p / "src" / "agent.py").exists()), None)
if PROJECT_DIR is None: raise FileNotFoundError("Make lawbot/ available or set LAWBOT_PROJECT_DIR.")
os.chdir(PROJECT_DIR); sys.path.insert(0, str(PROJECT_DIR))
from src.agent import LawBotAgent
from src.tools import TOOL_SCHEMAS
agent = LawBotAgent(project_dir=str(PROJECT_DIR))
print("Project:", PROJECT_DIR)
print(json.dumps(TOOL_SCHEMAS, indent=2))


## Tool examples

The case-law search uses the live API only when a token is present. The fallback searches the bundled Supreme Court summary paraphrases, never pretends to be live search, and retains the Court's notice that summaries are not judgments.


In [ ]:
tool_examples = [
    agent.run("Define mens rea."),
    agent.run("Calculate a deadline 10 business days from 2026-10-12 excluding holidays."),
    agent.run("Find case law about AI-generated fake citations in court decisions."),
]
for result in tool_examples:
    print("ROUTE:", result["route"], "GROUNDING:", result["grounding"])
    print("ANSWER:", result["answer"])
    print("TRACE:", json.dumps(result["tool_trace"], ensure_ascii=False, indent=2))
    print("-" * 80)


## RAG and direct-answer routes

If the Phase 3 FAISS index exists, retrieve up to five source snippets with scores. The demo exposes evidence and citations but does not synthesize a legal answer without an answer model. If no index exists, it abstains. The direct-answer callback is a placeholder and the agent labels it unverified.


In [ ]:
from src.agent import ABSTAIN
from src.rag import load_faiss_index, retrieve_with_mmr
index_dir = PROJECT_DIR / "data/rag_corpus/faiss_index"
state = {"vectorstore":None, "embeddings":None}
if (index_dir / "index.faiss").exists():
    # Load only the index produced locally by this project's Phase 3 pipeline.
    state["vectorstore"], state["embeddings"] = load_faiss_index(PROJECT_DIR)

def rag_handler(query, history=None):
    if state["vectorstore"] is None: return {"answer":ABSTAIN, "sources":[]}
    hits = retrieve_with_mmr(query, state["vectorstore"], state["embeddings"], k=5, fetch_k=20)
    sources = []
    for hit in hits:
        m = hit["document"].metadata
        sources.append({"source":m.get("act"), "title":m.get("doc_title"),
            "section":m.get("section_number") or m.get("section_numbers"), "page":m.get("page"),
            "snippet":hit["document"].page_content[:700], "score":round(hit["score"],4),
            "url":m.get("source_url")})
    return {"answer":"Retrieved candidate passages are listed with their sources. Review the original text before relying on them.", "sources":sources}

agent.rag_handler = rag_handler
rag_result = agent.run("What does Article 21 say about life and personal liberty?")
print("RAG ROUTE:", rag_result["route"], rag_result["grounding"])
print("ANSWER:", rag_result["answer"])
print("TRACE:", json.dumps(rag_result["tool_trace"], ensure_ascii=False, indent=2))

agent.direct_answer = lambda message, history=None: {"answer": "No answer model is connected in this trace-only example.", "grounding": "demo_placeholder"}
direct_result = agent.run("Hello, what can you help with?")
print("DIRECT ROUTE:", direct_result["route"], direct_result["grounding"])
print(direct_result["answer"])
all_results = tool_examples + [rag_result, direct_result]
trace_path = PROJECT_DIR / "reports/tool_trace_examples.json"
trace_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved example traces:", trace_path)
